# Graph and vector store — rebuild the VINE graph, add it to retrieval

**No LLM. No API key.** A T4 is enough; an A100 is faster.

What this notebook does, in order:

1. **Rebuilds the VINE graph** from the repo code and the manual. This build has:
   - the question-wording speed words removed (they came from a test question);
   - table rows no longer overwritten in Tables 2C-4, 4C-1 and 7B-1;
   - figure readings attached only to the figure they describe (not also to the
     table with the same number);
   - plural and range references linked ("Sections 2C.70 through 2C.73"), and the
     references inside list lead-ins ("Notes for Figure 6P-10 ...");
   - three figure readings corrected against the page (3B-28, 2E-21, 2E-31).
   The old graph is kept as `vine_data/graph_cache.before_2026-10-06.pkl`.
2. **Builds the items retrieval was missing**: one item per figure reading and one
   per table row. They come from the graph only — no question, no answer.
3. **Embeds them with BGE-M3** (the same encoder mode as the store) and adds them to
   the chunk collection, so search can find them.
4. **Saves the store to Drive.** The old snapshot is kept as
   `qdrant_db.before_vine_items.tar` until you delete it.

Then run `Retrieval_Test.ipynb` (CELL 1 to CELL 8). Its CELL 8 compares the new run
with the one before these items were added.

**Run every code cell in order: CELL 1 to CELL 6.**

In [ ]:
# CELL 1 — Drive, repo, packages. MRAG_BASE_DIR must be set BEFORE any mrag import.
import sys, os
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_DIR = "/content/drive/MyDrive/Beyond_RAG"
REPO_DIR  = "/content/Beyond_RAG_repo"
REPO_URL  = "https://github.com/hannanazad/Beyond_RAG.git"

os.environ["MRAG_ENV"]      = "colab"
os.environ["MRAG_BASE_DIR"] = DRIVE_DIR

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull
sys.path.insert(0, REPO_DIR)

!pip install -q --index-url https://download.pytorch.org/whl/cu124 torch==2.6.0 torchvision==0.21.0
!pip install -q -r $REPO_DIR/requirements.txt
!pip install -q --no-deps --force-reinstall "transformers>=4.49,<4.55" "huggingface_hub>=0.34,<0.35" "tokenizers>=0.21,<0.22" "torchao>=0.13,<0.14"
!pip install -q pymupdf

for f in ("mrag/vine/vector_items.py", "data/mutcd/FIGURE_READING_LOG.md"):
    assert os.path.exists(f"{REPO_DIR}/{f}"), f"{f} is not in the clone -- push the new files first"
print("data :", DRIVE_DIR)
print("code :", REPO_DIR)

In [ ]:
# CELL 2 — make sure the vector store is on local disk (same as Retrieval_Test CELL 2).
import shutil, subprocess, time
from pathlib import Path
from mrag.config import CFG

LOCAL = Path(CFG.qdrant_dir)
SNAP  = Path(CFG.base_dir) / "qdrant_db.tar"

def has_store(p: Path) -> bool:
    return (p / "collection" / CFG.coll_chunks).exists()

if has_store(LOCAL):
    print("vector store already on local disk:", LOCAL)
elif SNAP.exists():
    print(f"restoring {SNAP} ({SNAP.stat().st_size / 1e6:.0f} MB) ...")
    t0 = time.time()
    if LOCAL.exists():
        shutil.rmtree(LOCAL)
    subprocess.run(["tar", "-xf", str(SNAP), "-C", str(LOCAL.parent)], check=True)
    print(f"  done in {time.time() - t0:.0f}s")
else:
    raise FileNotFoundError(f"No vector store at {LOCAL} and no snapshot at {SNAP}.")
assert has_store(LOCAL), f"{LOCAL}/collection/{CFG.coll_chunks} is missing after the restore"
print("ok:", LOCAL)

### Rebuild the graph
Inputs: `chunks.jsonl` and `mutcd_tables.jsonl` from Drive, the MUTCD PDF, and the
three reading files **from the repo** (`data/mutcd/`), which carry the corrections.
Takes under a minute.

In [ ]:
# CELL 3 — rebuild the VINE graph and save it to Drive.
import json, pickle
from collections import Counter
from mrag.vine.graph_links import build as build_sentence_graph, FHWA_2_09_2
from mrag.vine import graph_enrich

DATA_DIR = Path(CFG.base_dir) / "vine_data"
GRAPH    = DATA_DIR / "graph_cache.pkl"
BACKUP   = DATA_DIR / "graph_cache.before_2026-10-06.pkl"
TABLES   = CFG.cache_dir / "mutcd_tables.jsonl"
LOGS     = Path(REPO_DIR) / "data" / "mutcd"
DATA_DIR.mkdir(parents=True, exist_ok=True)
assert TABLES.exists(), f"missing {TABLES} -- run CELL 6 of Beyond_RAG_VINE_Colab.ipynb once"
assert CFG.pdf_path.exists(), f"missing the MUTCD PDF at {CFG.pdf_path}"

t0 = time.time()
chunks = [json.loads(l) for l in open(CFG.chunks_jsonl) if l.strip()]
N, E, cand = build_sentence_graph(chunks, tables_path=str(TABLES), interpretations=[FHWA_2_09_2])
print(f"step 1: {len(N):,} nodes, {len(E):,} edges")
graph_enrich.enrich(N, E, chunks, pdf=CFG.pdf_path,
                    figure_log=LOGS / "FIGURE_READING_LOG.md",
                    rules=LOGS / "figure_rules.json",
                    reading_log=LOGS / "MANUAL_READING_LOG.md",
                    tables=TABLES)
ok, tot = graph_enrich.resolution(N, E)
print(f"step 2: {len(N):,} nodes, {len(E):,} edges, {100 * ok / tot:.2f}% of edges resolve "
      f"({time.time() - t0:.0f}s)")

# What the same build gave from the same inputs when it was written. A different
# number means different inputs (chunks, tables) or different code -- look before
# going on; it is not automatically wrong.
EXPECTED = (19671, 50406)
if (len(N), len(E)) != EXPECTED:
    print(f"NOTE: expected {EXPECTED[0]:,} nodes / {EXPECTED[1]:,} edges")

# The specific things this build fixes, checked on the graph itself.
out = {}
for e in E:
    out.setdefault(e.src, []).append(e)
checks = {
    "Table 2C-4 chart A and chart B rows are separate":
        "trow:Table 2C-4#s1-Ar0" in N and N["trow:Table 2C-4#s1-Br0"].text == "5 mph",
    "no figure reading sits on a table of the same number":
        not any(e.rel == "HAS_READING" and e.src == "figure:Table 2C-4" for e in E),
    "'see Sections 2C.70 through 2C.73' links all four":
        {"section:2C.70", "section:2C.71", "section:2C.72", "section:2C.73"}
        <= {e.dst for e in out.get("sent:MUTCD11e_2C17_Option_03#0", [])},
    "the Notes for Figure 6P-10 link to the figure":
        any(e.dst == "figure:Figure 6P-10" for e in out.get("lead:MUTCD11e_6P01_TA10_Option_01", [])),
    "the question-wording speed words are gone":
        "posted at" not in open(f"{REPO_DIR}/mrag/vine/graph_links.py").read(),
}
for name, passed in checks.items():
    print(f"  [{'ok' if passed else 'FAIL'}] {name}")
assert all(checks.values()), "a fix did not take -- is the clone up to date?"

if GRAPH.exists() and not BACKUP.exists():
    shutil.copy2(GRAPH, BACKUP)
    print("old graph kept as", BACKUP.name)
pickle.dump((N, E, chunks), open(GRAPH, "wb"))
os.environ["MUTCD_GRAPH"] = str(GRAPH)
print("saved", GRAPH)

In [ ]:
# CELL 4 — the items retrieval was missing: figure readings and table rows.
from mrag.vine.vector_items import build_items, write_items, summary, embed_text

items = build_items(GRAPH, TABLES)
ITEMS = DATA_DIR / "vine_items.jsonl"
write_items(items, ITEMS)
print(summary(items))
print("written to", ITEMS)
for want in ("FigureReading", "TableRow"):
    it = next(i for i in items if i["content_type"] == want)
    print(f"\n[{want}] {it['chunk_id']}\n  {embed_text(it)[:400]}")

### Add them to the vector store
The encoder must be the one the store was built with (the build manifest says which).
Items from an earlier run of this cell that are no longer produced are removed first,
so re-running is safe.

In [ ]:
# CELL 5 — embed with BGE-M3 and add to the chunk collection.
from qdrant_client.http import models as qm
from mrag.embeddings import TextEmbedder
from mrag.vector_store import VectorStore, ChunkRow, chunk_id_to_int

manifest = json.load(open(CFG.cache_dir / "text_embeddings_manifest.json"))
store_mode = manifest["chunks"]["encoder_mode"]
te = TextEmbedder(CFG.bge_m3_model).load()
print("encoder mode:", te._mode, "| store was built with:", store_mode)
assert te._mode == store_mode, "different encoder mode -- the new vectors would not match the old ones"

store = VectorStore(CFG.qdrant_dir)
n_file = sum(1 for _ in open(CFG.chunks_jsonl))
n_before = store.client.count(CFG.coll_chunks, exact=True).count

# items written by an earlier run of this cell
old_ids, offset = [], None
flt = qm.Filter(must=[qm.FieldCondition(key="source", match=qm.MatchValue(value="vine_graph"))])
while True:
    pts, offset = store.client.scroll(CFG.coll_chunks, scroll_filter=flt, limit=1000,
                                      offset=offset, with_payload=False, with_vectors=False)
    old_ids += [p.id for p in pts]
    if offset is None:
        break
new_ids = {chunk_id_to_int(i["chunk_id"]) for i in items}
stale = [pid for pid in old_ids if pid not in new_ids]
if stale:
    store.client.delete(CFG.coll_chunks, points_selector=qm.PointIdsList(points=stale), wait=True)
print(f"store before: {n_before:,} | manual chunks: {n_file:,} | earlier VINE items: {len(old_ids):,} "
      f"(removing {len(stale):,} no longer produced)")

t0 = time.time()
texts = [embed_text(i) for i in items]
dense, sparse = te.encode_both(texts, batch_size=32)
rows = [ChunkRow(id=chunk_id_to_int(i["chunk_id"]), dense=d, sparse=s, payload=i)
        for i, d, s in zip(items, dense, sparse)]
store.upsert_chunks(CFG.coll_chunks, rows)
n_after = store.client.count(CFG.coll_chunks, exact=True).count
empty_sparse = sum(1 for s in sparse if not s)
print(f"embedded {len(rows):,} items in {time.time() - t0:.0f}s | items with no sparse weights: {empty_sparse}")
print(f"store after: {n_after:,} = {n_file:,} manual chunks + {n_after - n_file:,} VINE items")
assert n_after == n_file + len(items), "the store count does not add up -- do not snapshot"
store.close()
del te

In [ ]:
# CELL 6 — save the store to Drive. The previous snapshot is kept once, by name.
TMP = Path(CFG.base_dir) / "qdrant_db.writing.tar"
OLD = Path(CFG.base_dir) / "qdrant_db.before_vine_items.tar"
t0 = time.time()
subprocess.run(["tar", "-cf", str(TMP), "-C", str(LOCAL.parent), LOCAL.name], check=True)
if SNAP.exists():
    if OLD.exists():
        SNAP.unlink()
    else:
        SNAP.rename(OLD)
        print("previous snapshot kept as", OLD.name)
TMP.rename(SNAP)
print(f"saved {SNAP} ({SNAP.stat().st_size / 1e6:.0f} MB) in {time.time() - t0:.0f}s")
print("\nNext: run Retrieval_Test.ipynb, CELL 1 to CELL 8.")